In [24]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [26]:
import cupy
print(cupy.cuda.runtime.getDeviceCount())  # 1


ImportError: 
================================================================
Failed to import CuPy.

If you installed CuPy via wheels (cupy-cudaXXX or cupy-rocm-X-X), make sure that the package matches with the version of CUDA or ROCm installed.

On Linux, you may need to set LD_LIBRARY_PATH environment variable depending on how you installed CUDA/ROCm.
On Windows, try setting CUDA_PATH environment variable.

Check the Installation Guide for details:
  https://docs.cupy.dev/en/latest/install.html

CUDA Path: None
DLL dependencies:
  KERNEL32.dll -> C:\WINDOWS\System32\KERNEL32.DLL
  MSVCP140.dll -> C:\WINDOWS\SYSTEM32\MSVCP140.dll
  VCRUNTIME140.dll -> C:\Users\PC\AppData\Local\Programs\Python\Python312\VCRUNTIME140.dll
  api-ms-win-crt-convert-l1-1-0.dll -> C:\WINDOWS\System32\ucrtbase.dll
  api-ms-win-crt-environment-l1-1-0.dll -> C:\WINDOWS\System32\ucrtbase.dll
  api-ms-win-crt-filesystem-l1-1-0.dll -> C:\WINDOWS\System32\ucrtbase.dll
  api-ms-win-crt-heap-l1-1-0.dll -> C:\WINDOW

In [25]:
gpu = spacy.prefer_gpu()
print("GPU disponible:", gpu)  # True = RTX 4060 usada

GPU disponible: False


In [ ]:
doc = nlp("Tea is healthy and calming, don't you think?")


In [ ]:
for token in doc:
    print(token)

### Explicación

Lemma es la base de palabra, por el ejemplo en este código **is** el lema es **be**, el lemma de **calming** es **calm** y el lemma de **don't** es **do**.

Las **stopword**, por otro lado, son las palabras cortas que no aportan nada al texto en este caso el ejemplo es **is, don't, you, and**.

In [ ]:
print(f"Token \t\tLemma \t\tStopword".format('Token', 'Lemma', 'Stopword'))
print("-"*40)
for token in doc:
    print(f"{str(token)}\t\t{token.lemma_}\t\t{token.is_stop}")

# Pattern Matching


In [ ]:
from spacy.matcher import PhraseMatcher
matcher = PhraseMatcher(nlp.vocab, attr='LOWER')

Crea un **buscador de frases (Phrase Matcher)** para que pueda detectar patrones dentro de un texto. Configurado para que las coincidencias sean insensibles a mayúsculas o minúsculas.

In [ ]:
terms = ['Galaxy Note', 'iPhone 11', 'iPhone XS', 'Google Pixel']
patterns = [nlp(text) for text in terms] # Pasa cada término por el pipeline nlp, el resultado son objetos DOC de spaCy(tokenizados, con su información lingüistica)
matcher.add("TerminologyList", patterns) #Registra estos patrones en el matcher con una etiqueta

Prepara un **listado de expresiones(frases)** para que un matcher de spaCy las pueda detectar luego dentro de otros textos.

**Idea mental:** Se construye un diccionario de frases y se lo enseñamos al matcher para que luego pueda subrayarlas cuando aparezcan dentro de textos más largos.

In [ ]:
# Borrowed from https://daringfireball.net/linked/2019/09/21/patel-11-pro
text_doc = nlp("Glowing review overall, and some really interesting side-by-side "
               "photography tests pitting the iPhone 11 Pro against the "
               "Galaxy Note 10 Plus and last year’s iPhone XS and Google Pixel 3.")
matches = matcher(text_doc)
print(matches)

Crea un documento spaCy a partir de un texto y luego busca dentro de ese texto todas las frases que previamente registré en el PhraseMatcher.

In [ ]:
match_id, start, end = matches[0]
print(nlp.vocab.strings[match_id], text_doc[start:end])

Toma la primera coincidencia e imprime la nombre de la regla del matcher y el texto encontrado.